<span style="text-align: center; color:#009999;">

# [LGBIO2050] - Medical Imaging
## *Non-Rigid Image Registration*
</span>

### Professors :   
- Prof. G. Kerckhofs  
- Prof. J. Lee
- Prof. B. Macq
- Prof. G. Duchêne
    
### Teaching assistants : 
- Pauline Hermans (pauline.hermans@uclouvain.be)
- Nicolas Delinte (nicolas.delinte@uclouvain.be)
- Simon Francois (simon.b.francois@uclouvain.be)

**2026-2027**

<span style="color:#009999;">
 
## 1. Guidelines and Deliverables
</span>

- This assignment is due on **11th November 2026**.
- The jupyter notebook containing the code and **detailed answers** to the questions must be delivered in an archive (.zip folder) on Moodle, with *"Markdown cells"* containing the answers to the questions and *"Code cells"* containing the code you implemented. The answers have to be written in English.
- This work must comply with the **EPL AI Guidelines** (available on Moodle) concerning the responsible use of AI in student work. Copying code or answers from other groups or from the internet is strictly prohibited. **Any source used for inspiration must be clearly acknowledged and referenced.**
- At the end of the semester, an **individual evaluation** will allow you to demonstrate that you have a good understanding of the challenge, both in terms of the code you have written and the answers to the reflection questions. Make sure that **all group members are familiar with each challenge and understand how it works**.
- The **presentation and overall formatting of your notebook** will also be taken into account in the assessment. Make sure to **write clean, efficient, well-structured code and comment it** appropriately to make it easy to understand. Your outputs and figures should also be clear and well-presented (do not forget titles, legends, etc. where appropriate). Pay particular attention to your answers to the questions as well. **Structure your explanations clearly** and use titles, subtitles, figures, and other relevant elements to make your **answers clear and precise**.  



<span style="color:#009999;">
 
## 2. Context
</span>


### Registration

The purpose of this challenge is to implement basic medical image registration tools. Aim of image registration is to align a ’moving’ image on a ’fixed’ or 'static' image by finding a spatial transformation mapping all pixels in the ’moving’ image to pixels in the ’fixed’ image.
Methods used in order to register medical images can be: 

- **Inter- vs Intra- patient.** Inter-patient methods register images captured on multiple patients while intra-patient methods register images captured on the same patient.
*In the context of this challenge, you will perform intra-patient image registration which is less sensitive to anatomical variations.*

- **Mono vs Multi- modal.** In the case of mono-modal registration, both ’fixed’ and ’moving’ images are of the same modality (MRI, PET, X-Rays,...). In the case of multi-modal registration, ’fixed’ and ’moving’ images are of different modalities (MRI/PET, PET/X-Rays,...). This second type of registration is useful to overlap different kind of information.
*In the context of this challenge, you will perform mono-modal image registration on CT-scans.*

- **Rigid vs non-rigid.** Rigid registration methods use a simple transform, uniformly applied over the entire image (like rotations or translations). In contrast, non-rigid registration methods allow to handle situations where the images to be registered have geometric differences that cannot be accounted by similarity (global translation, rotation, and scaling) transformations. In these cases, small variations in the images are taken into account in order to modify locally the deformation transform.
*In the context of this challenge, you will implement a non-rigid image registration tool.*


### Radiotherapy

Radiotherapy is a cancer treatment that uses ionizing radiation to destroy tumor cells while minimizing damage to surrounding healthy tissues. Before treatment, a planning CT scan is used to identify the tumor and determine the optimal distribution of the radiation dose, ensuring that the tumor receives the prescribed dose while limiting exposure to nearby organs. However, the patient's anatomy may change between treatment planning and actual radiation delivery due to breathing, changes in body position, or internal organ movement. As a result, the tumor may no longer be located exactly where it was during planning. Image registration is therefore essential to align the planning CT scan with an image acquired during treatment, allowing the tumor's current position to be identified and helping assess whether the planned radiation dose will still adequately cover the tumor.


<div class="alert alert-info">
    
**QUESTION 1**

In the following cell, import the images named ``CT_lung_planning.png`` and ``CT_lung_during_treatment.png``. Besides, import also the binary mask in ``Tumor_segmentation_for_planning.png`` file that gives the tumor's contour. Provide a preliminary visuallion of the problem by using the provided ``show_2images_and_difference`` function.

It is easy to see that these two images are not aligned. The purpose here is to automatically find the contour of the tumor on the moving image in order to assess the validity of the dose planning (i.e., by checking if the tumor is still entirely covered by the radiation beams). Thus, can you explain why registration is necessary in this context? Will the tumor be covered by the beam if its position is determined by the treatment planning?
</div>

*(Answer here)*

In [ ]:
import numpy as np
from skimage import measure
from skimage.io import imread
import matplotlib.pyplot as plt


def show_2images_and_difference(fixed, moving, mask=None):
    """
    Two image plot (fixed and moving) + difference (fixed-moving)

    Parameters
    ----------
    fixed : 2D array
        fixed image.
    moving : 2D array
        moving image.
    mask : 2D array, optional
        binary mask of the tumor. The default is None.

    Returns
    -------
    None
    """
    
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))

    for ax, img, title in zip(axes,[fixed, moving, fixed - moving],
                                ['Fixed image', 'Moving image', 'Difference']):
        ax.imshow(img, cmap='gray')
        ax.set_title(title, fontsize=14)
        ax.axis('off')

        if mask is not None and title != 'Difference':
            contour = measure.find_contours(mask.astype(float), 0)[0]
            ax.plot(contour[:, 1], contour[:, 0], 'y-', linewidth=0.5)

    plt.tight_layout()
    plt.show()
    plt.close()
    
# Plot the images (show_2images_and_difference)
show_2images_and_difference(fixed, moving, mask=tumor_mask)


<span style="color:#009999;">
 
## 3. Non-Rigid Registration
### 3.1 First... Ridig Registration!
</span>

Rigid registration is a way to transform images. Such modifications are simple transforms uniformly applied on the entire image (rotations, translations, etc.). Those methods are not discussed in this challenge but they are crutial as a preprocessing step before applying non-rigid registration.
Execute the cell below to translate the moving image onto the fixed one. 

In [ ]:
import numpy as np
from tqdm import tqdm
from scipy import ndimage


def SSD(im1, im2):
    """
    Compute the Sum of Squared Differences (SSD) between two images.

    Parameters
    ----------
    im1 : numpy.ndarray
        First image, represented as a numerical array.
    im2 : numpy.ndarray
        Second image, represented as a numerical array.

    Returns
    -------
    float or numpy scalar
        Sum of squared differences between the two images.
    """
    return np.sum((im1 - im2) ** 2)


def transform_image(moving, tx, ty, theta):
    """
    Apply a rigid transformation (rotation and translation) to the moving image.

    Parameters
    ----------
    moving : numpy.ndarray
        The moving image, represented as a numerical array.
    tx : float
        Horizontal translation in pixels.
    ty : float
        Vertical translation in pixels.
    theta : float
        Rotation angle in radians.

    Returns
    -------
    registered : numpy.ndarray
        Transformed image with the same shape as moving.
    """
    # Rotation around the image center
    rotated = ndimage.rotate(moving, np.degrees(theta),
                            reshape=False, order=1, mode='constant', cval=0)

    # Translation
    registered = ndimage.shift(rotated, shift=(ty, tx),
                            order=1, mode='constant', cval=0)

    return registered


def rigid_registration(fixed, moving, iter_max=100, step_trans=1.0, step_rot=0.01,
                       show_steps=True):
    """
    Register a moving image to a fixed image using rigid transformations
    and iterative optimization of the Sum of Squared Differences (SSD).

    Parameters
    ----------
    fixed : numpy.ndarray
        Reference image to which the moving image is registered.
    moving : numpy.ndarray
        Image to transform and align with the fixed image.
    iter_max : int, optional
        Maximum number of optimization iterations. Default is 100.
    step_trans : float, optional
        Initial translation step size in pixels. Default is 1.0.
    step_rot : float, optional
        Initial rotation step size in radians. Default is 0.01.
    show_steps : bool, optional
        Whether to display intermediate registration results. Default is True.

    Returns
    -------
    registered : numpy.ndarray
        Moving image after the estimated rigid transformation.
    tuple of float
        Estimated transformation parameters (tx, ty, theta).
    current_ssd : float or numpy scalar
        SSD between fixed and registered.
    """

    # 1. Initialization
    tx, ty, theta = 0.0, 0.0, 0.0

    registered = transform_image(moving, tx, ty, theta)
    current_ssd = SSD(fixed, registered)

    # 2. Iterative optimization
    pbar = tqdm(range(iter_max), desc='Iterative optimization')
    for i in pbar:

        improved = False

        # Test positive and negative changes in each parameter
        for dtx, dty, dtheta in [(step_trans, 0, 0), (-step_trans, 0, 0),
                                (0, step_trans, 0), (0, -step_trans, 0),
                                (0, 0, step_rot), (0, 0, -step_rot)]:

            new_tx = tx + dtx
            new_ty = ty + dty
            new_theta = theta + dtheta

            candidate = transform_image(moving, new_tx, new_ty, new_theta)
            candidate_ssd = SSD(fixed, candidate)

            # Keep the transformation if SSD decreases
            if candidate_ssd < current_ssd:
                tx, ty, theta = new_tx, new_ty, new_theta
                registered = candidate
                current_ssd = candidate_ssd
                improved = True
                #continue

        # If no improvement is found, reduce iteration steps size
        if not improved:
            step_trans /= 2
            step_rot /= 2

            # Stop if steps size become very small
            if step_trans < 0.005 and step_rot < 0.00005:
                break

        pbar.set_description(f"SSD = {current_ssd:.2f}, tx = {tx:.2f}, "
                            f"ty = {ty:.2f}, theta = {np.degrees(theta):.3f} degrees")

        if show_steps and i%10==0:
            show_2images_and_difference(fixed, registered, mask=tumor_mask)

    return registered, (tx, ty, theta), current_ssd


# Apply rigid registration
registered_rigid, params, SSD_val = rigid_registration(fixed, moving, iter_max=100, show_steps=True)

show_2images_and_difference(fixed, registered_rigid, mask=tumor_mask)

<div class="alert alert-info">
    
**QUESTION 2**

Comment the output of the rigid registration step on the moving image.
</div>

*(Answer here)*

<span style="color:#009999;">

### 3.2 Demons
</span>


The next step consists in applying non-rigid registration on these images by deforming the pre-treatment image in order to get the best possible matching between images (this is mainly an estimation of deformations due to breathing). Unlike rigid registration methods, the demons algorithm allows to compute local transforms. This means that the mapping is no more defined by only two parameters but instead by a deformation field. This field is computed in every pixel based on local information (for instance the intensity and the gradient of the image).

The demons method aims to approximately minimize the SSD as explained in [reference](https://pdfs.semanticscholar.org/29b2/4a6e4438993e5ec5fea139274a9570d430e3.pdf). The algorithm you will implement is a greedy algorithm composed of the following steps:

**1. Initialization**
  - Define a deformed image as being equal to the moving image
  - Initialize the accumulated deformation field
  - Compute the gradient in the 'fixed' image  <font color=#009999> *Hint: use the gradient function from numpy* </font>
 
**2. Iterative process**
 - Gradient descent step on the SSD criterion
     - Compute the gradient in the ’deformed’ image
     - Compute the update deformation field $\overrightarrow{F}$ at each pixel with the adapted demons equation:
 $$ \overrightarrow{F} = \frac{(S-M) \times (\nabla S + \nabla M)}
 {(S-M)^2 + ||\nabla S + \nabla M||^2} $$
where $M$ is the value of the pixel in the moving image, $S$ is the value of the pixel in the static image, $\overrightarrow{\nabla}M$ is the gradient of the moving image and $\overrightarrow{\nabla}S$ is the gradient of the static image. <font color=#009999>*Hint: compute the X and Y components of the field individually*</font>
 - Smoothing step
     - Regularize the field with a Gaussian smoothing performed by convoluting the two components of the field with a 2D Gaussian kernel. Try different variances to regularize your field and compare your results. <font color=#009999> *Hint: use the function gaussian_filter from scipy.ndimage* </font>
     - Add the update deformation field to the accumulated field
     - Calculate the new ’deformed’ image by interpolation
     - Compute the SSD metric


**Remarks :**
- Note that the <samp>show_2images_and_field</samp> function allows you to visualize the deformation field. 
- In order to deform the moving image, you will have to give to each pixel $(x, y)$ the value in the moving image
located at $(x_d, y_d ) = (x, y) + d(x, y)$, where $d(x, y)$ is the deformation vector at pixel $(x, y)$. If $(x_d , y_d )$ does not correspond to the coordinate of a pixel, you will have to interpolate the value between surrounding pixels. <font color=#009999> *Hint: use the function scipy.ndimage.map_coordinates* </font> <br>

<div class="alert alert-info">

**QUESTION 3**

Apply non-rigid registration on these images by deforming the moving image in order to get the best possible matching between images (this is mainly an estimation of deformations due to breathing). Comment the output of the non-rigid registration on the moving image.
</div>

*(Answer here)*

In [5]:

def show_2images_and_field(fixed, moving, field_x, field_y):
    """
    Two image plot (fixed and moving) + deformation field.

    Parameters
    ----------
    fixed : numpy.ndarray
        Fixed image.
    moving : numpy.ndarray
        Moving image.
    field_x : numpy.ndarray
        Deformation field in X coordinates.
    field_y : numpy.ndarray
        Deformation field in Y coordinates.

    Returns
    -------
    None
    """

    fig, axes = plt.subplots(1, 3, figsize=(9, 3))

    axes[0].imshow(fixed, cmap='gray')
    axes[0].set_title('Fixed image')

    axes[1].imshow(moving, cmap='gray')
    axes[1].set_title('Moving image')

    rows, cols = fixed.shape
    x, y = np.meshgrid(np.arange(cols), np.arange(rows))

    axes[2].quiver(
        x[::15, ::15], y[::15, ::15],
        -field_x[::15, ::15], -field_y[::15, ::15],
        angles='xy', scale_units='xy', scale=0.2
    )
    axes[2].set_title('Displacement field')
    axes[2].set_xlim(0, cols)
    axes[2].set_ylim(rows, 0)
    axes[2].set_aspect('equal')

    for ax in axes:
        ax.axis('off')

    plt.tight_layout()
    plt.show()



<div class="alert alert-info">

**QUESTION 4**

Which value of the variances (Gaussian kernel) seems to give the best result? Analyze the impact of a small/large value. Intuitively, does it make sense?
</div>

*(Answer here)*

<div class="alert alert-info">

**QUESTION 5**

Also, try to apply the demons on the original images (with no rigid registration) and comment your results. Is the final result different? Why?
</div>

*(Answer here)*

In [ ]:
# Apply Demons registration on intial moving image and show result (show_2images_and_difference)
